# Imitation Learning in MiniGrid DoorKey

This notebook builds a complete imitation-learning project around `MiniGrid-DoorKey-16x16-v0`. This larger DoorKey variant is still practical on CPU, but it gives behavior cloning much more room to drift into off-expert states.

We use three learners:

- behavior cloning (BC), trained only on expert demonstrations
- DAgger, which asks the expert to label states visited by the learner
- budgeted critical-state DAgger, which spends fewer expert labels on high-uncertainty learner states

All code lives in this notebook. Generated GIFs and plots are saved under `outputs/`.


In [ ]:
# Lightweight dependency cell for Colab, Kaggle, and local notebooks.
# It only installs packages that are missing in the current kernel.
import importlib.util
import subprocess
import sys

required_packages = {
    "gymnasium": "gymnasium",
    "minigrid": "minigrid",
    "torch": "torch",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "imageio": "imageio",
    "tqdm": "tqdm",
}

missing = [pip_name for import_name, pip_name in required_packages.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")


## Project knobs

We use moderate defaults for the larger DoorKey map. The full run is still CPU-friendly, but the larger map and longer rollouts make the imitation-learning differences easier to see.


In [ ]:
import copy
import math
import os
import random
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import Image as IPyImage
from IPython.display import Markdown, display
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import minigrid  # Registers MiniGrid environments with Gymnasium.
from minigrid.wrappers import ViewSizeWrapper

# Environment and output configuration.
ENV_ID = "MiniGrid-DoorKey-16x16-v0"
GRID_SIZE = 16
AGENT_VIEW_SIZE = 15
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reference-scale experiment settings. These intentionally match the scale of
# the supplied notebook outputs rather than the earlier quick-smoke settings.
SEED = 7
NUM_EXPERT_EPISODES = 1500
MAX_STEPS_PER_EPISODE = 768

BATCH_SIZE = 256
BC_EPOCHS = 20
BC_LR = 1e-3
VAL_FRACTION = 0.10

EVAL_EPISODES = 100

DAGGER_ITERS = 8
DAGGER_ROLLOUTS_PER_ITER = 10
DAGGER_EPOCHS_PER_ITER = 5
DAGGER_LR = 3e-4

CRITICAL_ITERS = 8
CRITICAL_ROLLOUTS_PER_ITER = 40
CRITICAL_LABELS_PER_ITER = 1200
CRITICAL_EPOCHS_PER_ITER = 5
CRITICAL_LR = 3e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
        torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


set_global_seed(SEED)
print(f"Using device: {DEVICE}")
print(f"Outputs will be written to: {OUTPUT_DIR.resolve()}")

## 1. Loading, exploring, and visualizing the environment

We first load the environment and define small compatibility helpers. MiniGrid observations are dictionaries; the main observation used by the learner is the symbolic image tensor with shape `(7, 7, 3)`. The three channels encode object type, color, and object state.

The action space is discrete. The useful low-level actions here are left, right, forward, pickup, and toggle. Rewards are sparse: reaching the goal gives a positive reward, while most intermediate steps give zero. An episode terminates when the agent reaches the goal or when the time limit is reached.


In [ ]:
# This is a helper function to create the MiniGrid environment. You may use it later. No need to change it.
def make_env(render_mode="rgb_array"):
    try:
        env = gym.make(ENV_ID, render_mode=render_mode)
    except TypeError:
        env = gym.make(ENV_ID)
    if AGENT_VIEW_SIZE != 7:
        env = ViewSizeWrapper(env, agent_view_size=AGENT_VIEW_SIZE)
    return env


# This is a helper function to reset Gymnasium or older Gym-style environments. You may use it later. No need to change it.
def reset_env(env, seed=None):
    if seed is None:
        result = env.reset()
    else:
        result = env.reset(seed=seed)
    if isinstance(result, tuple) and len(result) == 2:
        return result
    return result, {}


# This is a helper function to step Gymnasium or older Gym-style environments. You may use it later. No need to change it.
def step_env(env, action):
    result = env.step(int(action))
    if isinstance(result, tuple) and len(result) == 5:
        obs, reward, terminated, truncated, info = result
        done = bool(terminated or truncated)
        return obs, float(reward), done, info
    obs, reward, done, info = result
    return obs, float(reward), bool(done), info


# This is a helper function to read the MiniGrid image observation. You may use it later. No need to change it.
def obs_image(obs):
    if isinstance(obs, dict) and "image" in obs:
        return np.asarray(obs["image"], dtype=np.uint8)
    return np.asarray(obs, dtype=np.uint8)


# This is a helper function to render an RGB frame from the environment. You may use it later. No need to change it.
def render_env(env, tile_size=16):
    # Prefer get_frame so GIFs remain compact even on a 16x16 map.
    if hasattr(env.unwrapped, "get_frame"):
        frame = env.unwrapped.get_frame(tile_size=tile_size, agent_pov=False)
    else:
        try:
            frame = env.render()
        except TypeError:
            frame = env.render(mode="rgb_array")
    if frame is None:
        raise RuntimeError("The environment did not return an RGB frame.")
    return np.asarray(frame, dtype=np.uint8)


# This is a helper function to save frames as a GIF. You may use it later. No need to change it.
def save_gif(frames, path, fps=6):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    duration = 1.0 / fps
    imageio.mimsave(path, frames, duration=duration, loop=0)
    return path


# This is a helper function to display a generated GIF inline. You may use it later. No need to change it.
def display_gif(path):
    display(IPyImage(filename=str(path)))


# This is a helper function to read the agent position. You may use it later. No need to change it.
def get_agent_pos(env):
    pos = env.unwrapped.agent_pos
    return (int(pos[0]), int(pos[1]))


def rollout_policy(policy_fn, seed, gif_path=None, max_steps=MAX_STEPS_PER_EPISODE, fps=6):
    """Run one complete closed-loop episode.

    Positions include the initial state and every post-action state. Episode length is
    exactly the number of executed environment actions. A positive reward marks success.
    """
    env = make_env(render_mode="rgb_array")
    frames = []
    positions = []
    actions = []
    rewards = []
    terminated_flags = []
    truncated_flags = []

    try:
        obs, _ = reset_env(env, seed=seed)
        positions.append(get_agent_pos(env))
        if gif_path is not None:
            frames.append(render_env(env))

        success = False
        final_terminated = False
        final_truncated = False

        for step_idx in range(int(max_steps)):
            action = int(policy_fn(obs, env))
            result = env.step(action)

            if isinstance(result, tuple) and len(result) == 5:
                next_obs, reward, terminated, truncated, info = result
            else:
                next_obs, reward, done, info = result
                terminated, truncated = bool(done), False

            reward = float(reward)
            terminated = bool(terminated)
            truncated = bool(truncated)

            # Treat the notebook's common evaluation horizon as a truncation too.
            if step_idx + 1 >= int(max_steps) and not (terminated or truncated):
                truncated = True

            actions.append(action)
            rewards.append(reward)
            terminated_flags.append(terminated)
            truncated_flags.append(truncated)
            positions.append(get_agent_pos(env))

            if gif_path is not None:
                frames.append(render_env(env))

            success = bool(success or reward > 0.0)
            obs = next_obs
            final_terminated = terminated
            final_truncated = truncated

            if terminated or truncated:
                break

        saved_path = None
        if gif_path is not None:
            saved_path = save_gif(frames, gif_path, fps=fps)

        return {
            "seed": int(seed),
            "return": float(np.sum(rewards, dtype=np.float64)),
            "success": bool(success),
            "length": int(len(actions)),
            "positions": np.asarray(positions, dtype=np.int16),
            "actions": np.asarray(actions, dtype=np.int64),
            "rewards": np.asarray(rewards, dtype=np.float32),
            "terminated_flags": np.asarray(terminated_flags, dtype=bool),
            "truncated_flags": np.asarray(truncated_flags, dtype=bool),
            "terminated": bool(final_terminated),
            "truncated": bool(final_truncated),
            "gif_path": saved_path,
        }
    finally:
        env.close()


def evaluate_policy(policy_fn, n_episodes=EVAL_EPISODES, seed_start=0,
                    max_steps=MAX_STEPS_PER_EPISODE, record_positions=False):
    episode_returns = []
    episode_successes = []
    episode_lengths = []
    all_positions = []
    seeds = []

    for episode_idx in tqdm(range(int(n_episodes)), desc="evaluation", leave=False):
        seed = int(seed_start + episode_idx)
        rollout = rollout_policy(
            policy_fn,
            seed=seed,
            max_steps=max_steps,
        )
        seeds.append(seed)
        episode_returns.append(rollout["return"])
        episode_successes.append(rollout["success"])
        episode_lengths.append(rollout["length"])
        if record_positions:
            all_positions.extend(rollout["positions"].tolist())

    returns = np.asarray(episode_returns, dtype=np.float32)
    successes = np.asarray(episode_successes, dtype=bool)
    lengths = np.asarray(episode_lengths, dtype=np.int32)
    positions = np.asarray(all_positions, dtype=np.int16).reshape(-1, 2) if all_positions else np.empty((0, 2), dtype=np.int16)

    return {
        "seeds": np.asarray(seeds, dtype=np.int64),
        "episode_returns": returns,
        "episode_successes": successes,
        "episode_lengths": lengths,
        "success_rate": float(successes.mean()) if len(successes) else 0.0,
        "avg_return": float(returns.mean()) if len(returns) else 0.0,
        "avg_length": float(lengths.mean()) if len(lengths) else 0.0,
        "positions": positions,
    }


# This is a helper function to plot training curves. You may use it later. No need to change it.
def plot_training_curves(history, title="Training curves"):
    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    ax1.plot(epochs, history["train_loss"], label="train loss", color="tab:blue")
    if "val_loss" in history and len(history["val_loss"]) > 0:
        ax1.plot(epochs, history["val_loss"], label="val loss", color="tab:orange")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("cross-entropy loss")
    ax1.grid(alpha=0.25)

    if "val_acc" in history and len(history["val_acc"]) > 0:
        ax2 = ax1.twinx()
        ax2.plot(epochs, np.asarray(history["val_acc"]) * 100.0,
                 label="val accuracy", color="tab:green")
        ax2.set_ylabel("validation accuracy (%)")
        lines, labels = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines + lines2, labels + labels2, loc="center right")
    else:
        ax1.legend(loc="best")

    ax1.set_title(title)
    plt.tight_layout()
    slug = "_".join(title.lower().replace("-", " ").split())
    fig.savefig(OUTPUT_DIR / f"{slug}.png", dpi=160, bbox_inches="tight")
    plt.show()


# This is a helper function to plot a state visitation heatmap. You may use it later. No need to change it.
def plot_visit_heatmap(positions, width=GRID_SIZE, height=GRID_SIZE, ax=None, title="Visited states",
                       cmap="viridis", show_colorbar=True):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    counts = np.zeros((height, width), dtype=np.float32)
    for x, y in np.asarray(positions, dtype=np.int64):
        if 0 <= x < width and 0 <= y < height:
            counts[y, x] += 1.0

    im = ax.imshow(counts, origin="upper", cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xticks(range(width))
    ax.set_yticks(range(height))
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(which="major", color="white", linewidth=0.6, alpha=0.35)
    if show_colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax, counts


In [ ]:
inspect_env = make_env(render_mode="rgb_array")
inspect_obs, _ = reset_env(inspect_env, seed=SEED)

print("Environment:", ENV_ID)
print("Observation image shape:", obs_image(inspect_obs).shape)
print("Observation image dtype:", obs_image(inspect_obs).dtype)
print("Mission:", inspect_obs.get("mission", "") if isinstance(inspect_obs, dict) else "")
print("Initial agent position:", get_agent_pos(inspect_env))
print("Initial agent direction:", int(inspect_env.unwrapped.agent_dir))
print("Action space:", inspect_env.action_space)
if hasattr(inspect_env.unwrapped, "actions"):
    print("Actions:", [(int(action), action.name) for action in inspect_env.unwrapped.actions])

inspect_frame = render_env(inspect_env)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(inspect_frame)
ax.set_title(f"{ENV_ID}, seed={SEED}")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "seeded_environment.png", dpi=160, bbox_inches="tight")
plt.show()
inspect_env.close()

In [ ]:
# Run and visualize a short random-policy baseline using only useful DoorKey actions.
RANDOM_ALLOWED_ACTIONS = np.asarray([0, 1, 2, 3, 5], dtype=np.int64)


def make_random_policy(seed):
    rng = np.random.default_rng(seed)

    def policy(obs, env):
        del obs, env
        return int(rng.choice(RANDOM_ALLOWED_ACTIONS))

    return policy


random_horizon = min(40, MAX_STEPS_PER_EPISODE)
random_eval = evaluate_policy(
    make_random_policy(SEED + 10),
    n_episodes=8,
    seed_start=100,
    max_steps=random_horizon,
    record_positions=True,
)
random_rollout = rollout_policy(
    make_random_policy(SEED + 11),
    seed=101,
    gif_path=OUTPUT_DIR / "random_policy.gif",
    max_steps=random_horizon,
)

print(f"Random-policy success rate: {100 * random_eval['success_rate']:.1f}%")
print(f"Random-policy average return: {random_eval['avg_return']:.3f}")
print(f"Random-policy average length: {random_eval['avg_length']:.1f}")
display_gif(random_rollout["gif_path"])

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_visit_heatmap(
    random_eval["positions"],
    ax=ax,
    title="Random-policy visitation heatmap",
    cmap="Greys",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "random_policy_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 2. Generating expert demonstrations with a BFS oracle

We now build an expert planner from scratch. The oracle is allowed to use privileged full-state access to the grid layout. It plans over symbolic states:

- agent position `(x, y)`
- agent direction
- whether the key is currently carried
- whether the door is open or locked

The plan is still executed with valid low-level MiniGrid actions: turn left, turn right, move forward, pick up, and toggle the door.


In [ ]:
LEFT, RIGHT, FORWARD, PICKUP, DROP, TOGGLE, DONE = range(7)
DIR_TO_VEC = {
    0: (1, 0),    # east
    1: (0, 1),    # south
    2: (-1, 0),   # west
    3: (0, -1),   # north
}
ORACLE_ACTIONS = [FORWARD, PICKUP, TOGGLE, LEFT, RIGHT]


@dataclass(frozen=True)
class DoorKeyLayout:
    width: int
    height: int
    walls: frozenset
    key_pos: tuple
    door_pos: tuple
    goal_pos: tuple


def build_layout(env):
    grid = env.unwrapped.grid
    walls = set()
    key_pos, door_pos, goal_pos = None, None, None

    for x in range(int(grid.width)):
        for y in range(int(grid.height)):
            cell = grid.get(x, y)
            if cell is None:
                continue
            if cell.type == "wall":
                walls.add((int(x), int(y)))
            elif cell.type == "key":
                key_pos = (int(x), int(y))
            elif cell.type == "door":
                door_pos = (int(x), int(y))
            elif cell.type == "goal":
                goal_pos = (int(x), int(y))

    missing = [
        name for name, value in {
            "key": key_pos,
            "door": door_pos,
            "goal": goal_pos,
        }.items() if value is None
    ]
    if missing:
        raise RuntimeError(
            "Could not extract the complete DoorKey layout. "
            f"Missing objects: {missing}. Build the layout immediately after reset."
        )

    return DoorKeyLayout(
        width=int(grid.width),
        height=int(grid.height),
        walls=frozenset(walls),
        key_pos=key_pos,
        door_pos=door_pos,
        goal_pos=goal_pos,
    )


def symbolic_state_from_env(env, layout):
    x, y = get_agent_pos(env)
    direction = int(env.unwrapped.agent_dir)
    carrying = env.unwrapped.carrying
    has_key = bool(carrying is not None and getattr(carrying, "type", None) == "key")

    door = env.unwrapped.grid.get(*layout.door_pos)
    if door is None or getattr(door, "type", None) != "door":
        raise RuntimeError("Door object is missing from its fixed layout position.")
    door_open = bool(door.is_open)
    door_locked = bool(door.is_locked)

    return (int(x), int(y), direction, has_key, door_open, door_locked)


# This is a helper function to compute the cell in front of a symbolic state. You may use it later. No need to change it.
def front_cell(state):
    x, y, direction, has_key, door_open, door_locked = state
    dx, dy = DIR_TO_VEC[int(direction)]
    return (x + dx, y + dy)


# This is a helper function to test whether a symbolic state can enter a grid cell. You may use it later. No need to change it.
def can_enter_cell(layout, pos, has_key, door_open):
    x, y = pos
    if x < 0 or x >= layout.width or y < 0 or y >= layout.height:
        return False
    if pos in layout.walls:
        return False
    if pos == layout.door_pos:
        return bool(door_open)
    if pos == layout.key_pos and not has_key:
        return False
    return True


def symbolic_transition(layout, state, action):
    x, y, direction, has_key, door_open, door_locked = state
    action = int(action)

    if action == LEFT:
        direction = (direction - 1) % 4
    elif action == RIGHT:
        direction = (direction + 1) % 4
    elif action == FORWARD:
        candidate = front_cell(state)
        if can_enter_cell(layout, candidate, has_key, door_open):
            x, y = candidate
    elif action == PICKUP:
        if front_cell(state) == layout.key_pos and not has_key:
            has_key = True
    elif action == TOGGLE:
        if front_cell(state) == layout.door_pos:
            if door_locked:
                if has_key:
                    door_locked = False
                    door_open = True
            else:
                door_open = not door_open

    return (int(x), int(y), int(direction), bool(has_key), bool(door_open), bool(door_locked))


# This is a helper function to recover an action sequence from BFS parent pointers. You may use it later. No need to change it.
def reconstruct_plan(parent, action_taken, goal_state):
    actions = []
    state = goal_state
    while parent[state] is not None:
        actions.append(action_taken[state])
        state = parent[state]
    actions.reverse()
    return actions


def oracle_plan_from_state(layout, start_state):
    start_state = tuple(start_state)
    if (start_state[0], start_state[1]) == layout.goal_pos:
        return []

    queue = deque([start_state])
    parent = {start_state: None}
    action_taken = {}

    while queue:
        state = queue.popleft()
        for action in ORACLE_ACTIONS:
            next_state = symbolic_transition(layout, state, action)
            if next_state in parent:
                continue
            parent[next_state] = state
            action_taken[next_state] = int(action)

            if (next_state[0], next_state[1]) == layout.goal_pos:
                return reconstruct_plan(parent, action_taken, next_state)
            queue.append(next_state)

    return None


# This is a helper function to query the expert action for the current environment state. You may use it later. No need to change it.
def oracle_action_from_env(env, layout):
    state = symbolic_state_from_env(env, layout)
    plan = oracle_plan_from_state(layout, state)
    if plan is None:
        raise RuntimeError(f"No valid oracle plan exists from state {state}.")
    if len(plan) == 0:
        return DONE
    return int(plan[0])


# This is a helper function to query the expert action for a stored symbolic state. You may use it later. No need to change it.
def oracle_action_from_symbolic_state(layout, state):
    plan = oracle_plan_from_state(layout, state)
    if plan is None:
        raise RuntimeError(f"No valid oracle plan exists from stored state {state}.")
    if len(plan) == 0:
        return DONE
    return int(plan[0])


def make_oracle_policy():
    layout_cache = {}

    def oracle_policy(obs, env):
        del obs
        env_key = id(env.unwrapped)
        step_count = int(getattr(env.unwrapped, "step_count", 0))
        if env_key not in layout_cache or step_count == 0:
            layout_cache[env_key] = build_layout(env)
        return oracle_action_from_env(env, layout_cache[env_key])

    return oracle_policy

In [ ]:
oracle_policy = make_oracle_policy()
ORACLE_SMOKE_EPISODES = 25
oracle_smoke = evaluate_policy(
    oracle_policy,
    n_episodes=ORACLE_SMOKE_EPISODES,
    seed_start=500,
    max_steps=MAX_STEPS_PER_EPISODE,
)

print(f"Oracle smoke-test success: {oracle_smoke['success_rate'] * 100:.1f}%")
print(f"Oracle smoke-test average length: {oracle_smoke['avg_length']:.1f}")
assert oracle_smoke["success_rate"] == 1.0, (
    "The BFS oracle must solve every smoke-test instance before dataset generation."
)

In [ ]:
def generate_expert_dataset(n_episodes, seed_start=1000, max_steps=MAX_STEPS_PER_EPISODE):
    all_obs = []
    all_actions = []
    all_positions = []
    all_rewards = []
    all_terminated = []
    all_truncated = []
    all_episode_ids = []
    all_timesteps = []
    all_symbolic_states = []
    transition_returns = []
    transition_lengths = []
    transition_successes = []

    episode_returns = []
    episode_lengths = []
    episode_successes = []
    episode_seeds = []

    for episode_id in tqdm(range(int(n_episodes)), desc="expert demonstrations"):
        env = make_env(render_mode="rgb_array")
        seed = int(seed_start + episode_id)
        try:
            obs, _ = reset_env(env, seed=seed)
            layout = build_layout(env)

            ep_obs = []
            ep_actions = []
            ep_positions = []
            ep_rewards = []
            ep_terminated = []
            ep_truncated = []
            ep_states = []
            ep_return = 0.0
            ep_success = False

            for timestep in range(int(max_steps)):
                state = symbolic_state_from_env(env, layout)
                expert_action = oracle_action_from_symbolic_state(layout, state)
                if expert_action == DONE:
                    break

                # Store the learner-visible observation immediately before the expert action.
                ep_obs.append(obs_image(obs).copy())
                ep_actions.append(int(expert_action))
                ep_positions.append(get_agent_pos(env))
                ep_states.append(state)

                result = env.step(int(expert_action))
                if isinstance(result, tuple) and len(result) == 5:
                    next_obs, reward, terminated, truncated, info = result
                else:
                    next_obs, reward, done, info = result
                    terminated, truncated = bool(done), False

                reward = float(reward)
                terminated = bool(terminated)
                truncated = bool(truncated)
                if timestep + 1 >= int(max_steps) and not (terminated or truncated):
                    truncated = True

                ep_rewards.append(reward)
                ep_terminated.append(terminated)
                ep_truncated.append(truncated)
                ep_return += reward
                ep_success = bool(ep_success or reward > 0.0)
                obs = next_obs

                if terminated or truncated:
                    break

            ep_length = len(ep_actions)
            if not ep_success:
                raise RuntimeError(
                    f"The oracle failed on demonstration seed {seed}; "
                    "dataset generation was stopped."
                )

            all_obs.extend(ep_obs)
            all_actions.extend(ep_actions)
            all_positions.extend(ep_positions)
            all_rewards.extend(ep_rewards)
            all_terminated.extend(ep_terminated)
            all_truncated.extend(ep_truncated)
            all_episode_ids.extend([episode_id] * ep_length)
            all_timesteps.extend(range(ep_length))
            all_symbolic_states.extend(ep_states)
            transition_returns.extend([ep_return] * ep_length)
            transition_lengths.extend([ep_length] * ep_length)
            transition_successes.extend([ep_success] * ep_length)

            episode_returns.append(ep_return)
            episode_lengths.append(ep_length)
            episode_successes.append(ep_success)
            episode_seeds.append(seed)
        finally:
            env.close()

    return {
        "obs": np.asarray(all_obs, dtype=np.uint8),
        "actions": np.asarray(all_actions, dtype=np.int64),
        "positions": np.asarray(all_positions, dtype=np.int16),
        "rewards": np.asarray(all_rewards, dtype=np.float32),
        "terminated": np.asarray(all_terminated, dtype=bool),
        "truncated": np.asarray(all_truncated, dtype=bool),
        "episode_ids": np.asarray(all_episode_ids, dtype=np.int32),
        "timesteps": np.asarray(all_timesteps, dtype=np.int32),
        "symbolic_states": np.asarray(all_symbolic_states, dtype=object),
        "returns": np.asarray(transition_returns, dtype=np.float32),
        "lengths": np.asarray(transition_lengths, dtype=np.int32),
        "successes": np.asarray(transition_successes, dtype=bool),
        "episode_returns": np.asarray(episode_returns, dtype=np.float32),
        "episode_lengths": np.asarray(episode_lengths, dtype=np.int32),
        "episode_successes": np.asarray(episode_successes, dtype=bool),
        "episode_seeds": np.asarray(episode_seeds, dtype=np.int64),
    }


# Create, summarize, and visualize the expert demonstration dataset.
expert_data = generate_expert_dataset(
    NUM_EXPERT_EPISODES,
    seed_start=1000,
    max_steps=MAX_STEPS_PER_EPISODE,
)

print("Expert transitions:", len(expert_data["actions"]))
print("Expert episodes:", NUM_EXPERT_EPISODES)
print("Expert success rate:", expert_data["episode_successes"].mean())
print("Expert average length:", expert_data["episode_lengths"].mean().round(2))
print("Stored terminated flags:", int(expert_data["terminated"].sum()))
print("Stored truncated flags:", int(expert_data["truncated"].sum()))
print("Expert observation tensor shape:", expert_data["obs"].shape)
assert expert_data["obs"].shape[1:] == (AGENT_VIEW_SIZE, AGENT_VIEW_SIZE, 3)
assert expert_data["episode_successes"].all(), "Every expert demonstration must succeed."

expert_gif = rollout_policy(
    make_oracle_policy(),
    seed=1000,
    gif_path=OUTPUT_DIR / "expert_rollout.gif",
)
display_gif(expert_gif["gif_path"])

fig, ax = plt.subplots(figsize=(4.8, 4.8))
plot_visit_heatmap(
    expert_data["positions"],
    ax=ax,
    title="Expert visitation heatmap",
    cmap="YlGnBu",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "expert_visitation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Training a behavior cloning policy

Behavior cloning turns imitation learning into supervised learning. We train a policy to predict the expert action from the MiniGrid image observation.

$$
L_{BC}(\theta) = -\mathbb{E}_{(s,a)\sim D_E}\log \pi_\theta(a\mid s).
$$

We use a deliberately small CNN. The observation is symbolic rather than photographic, so a compact model is enough for this project.


In [ ]:
OBS_NORMALIZER = torch.tensor([10.0, 5.0, 2.0], dtype=torch.float32).view(3, 1, 1)
N_ACTIONS = 7
ALLOWED_ACTIONS = (LEFT, RIGHT, FORWARD, PICKUP, TOGGLE)

# This is a helper function to preprocess one MiniGrid image for the model. You may use it later. No need to change it.
def preprocess_image(image):
    x = torch.as_tensor(image, dtype=torch.float32)
    if x.ndim != 3:
        raise ValueError(f"Expected image with 3 dimensions, got shape {tuple(x.shape)}")
    if x.shape[-1] == 3:
        x = x.permute(2, 0, 1)
    if x.shape[0] != 3:
        raise ValueError(f"Expected three semantic channels, got shape {tuple(x.shape)}")
    return x / OBS_NORMALIZER.to(x.device)


# This is a helper function to turn an observation into a model input batch. You may use it later. No need to change it.
def obs_to_tensor(obs, device=DEVICE):
    x = preprocess_image(obs_image(obs)).unsqueeze(0)
    return x.to(device)


class ImageActionDataset(Dataset):

    def __init__(self, images, actions, indices=None):
        self.images = np.asarray(images, dtype=np.uint8)
        self.actions = np.asarray(actions, dtype=np.int64)
        if len(self.images) != len(self.actions):
            raise ValueError("images and actions must contain the same number of examples")
        if indices is None:
            indices = np.arange(len(self.actions), dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return int(len(self.indices))

    def __getitem__(self, index):
        source_index = int(self.indices[index])
        image = preprocess_image(self.images[source_index])
        action = torch.tensor(self.actions[source_index], dtype=torch.long)
        return image, action


def make_data_loaders(images, actions, val_fraction=VAL_FRACTION,
                      batch_size=BATCH_SIZE, seed=SEED):
    n_examples = int(len(actions))
    if n_examples < 2:
        raise ValueError("At least two examples are required for a train/validation split.")

    rng = np.random.default_rng(seed)
    permutation = rng.permutation(n_examples)
    n_val = int(round(float(val_fraction) * n_examples))
    n_val = min(max(n_val, 1), n_examples - 1)
    val_indices = permutation[:n_val]
    train_indices = permutation[n_val:]

    train_dataset = ImageActionDataset(images, actions, train_indices)
    val_dataset = ImageActionDataset(images, actions, val_indices)

    generator = torch.Generator()
    generator.manual_seed(int(seed))
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(batch_size),
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader


class SmallMinigridPolicy(nn.Module):

    def __init__(self):
        super().__init__()
        pooled_size = AGENT_VIEW_SIZE // 2
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * pooled_size * pooled_size, 128),
            nn.ReLU(),
            nn.Dropout(p=0.10),
            nn.Linear(128, N_ACTIONS),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_policy(model, train_loader, val_loader=None, epochs=5, lr=1e-3,
                 device=DEVICE, desc="training"):
    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_state = None
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 0

    model.to(device)
    for epoch in tqdm(range(int(epochs)), desc=desc):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, actions in train_loader:
            images = images.to(device, non_blocking=True)
            actions = actions.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = F.cross_entropy(logits, actions)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            batch_size = int(actions.shape[0])
            train_loss_sum += float(loss.item()) * batch_size
            train_count += batch_size

        train_loss = train_loss_sum / max(train_count, 1)
        history["train_loss"].append(train_loss)

        if val_loader is not None:
            model.eval()
            val_loss_sum = 0.0
            val_correct = 0
            val_count = 0
            with torch.no_grad():
                for images, actions in val_loader:
                    images = images.to(device, non_blocking=True)
                    actions = actions.to(device, non_blocking=True)
                    logits = model(images)
                    loss = F.cross_entropy(logits, actions)
                    batch_size = int(actions.shape[0])
                    val_loss_sum += float(loss.item()) * batch_size
                    val_correct += int((logits.argmax(dim=1) == actions).sum().item())
                    val_count += batch_size

            val_loss = val_loss_sum / max(val_count, 1)
            val_acc = val_correct / max(val_count, 1)
        else:
            val_loss = train_loss
            val_acc = 0.0

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch + 1
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

    if best_state is None:
        raise RuntimeError("Training did not produce a checkpoint.")
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    history["best_epoch"] = int(best_epoch)
    history["best_val_loss"] = float(best_val_loss)
    history["best_val_acc"] = float(best_val_acc)
    print(
        f"{desc}: restored epoch {best_epoch} "
        f"(validation loss={best_val_loss:.4f}, accuracy={100 * best_val_acc:.1f}%)"
    )
    return history


@torch.no_grad()
def model_action(model, obs, deterministic=True, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    model.eval()
    logits = model(obs_to_tensor(obs, next(model.parameters()).device))[0]
    allowed = torch.tensor(ALLOWED_ACTIONS, dtype=torch.long, device=logits.device)
    allowed_logits = logits.index_select(0, allowed) / float(temperature)

    if deterministic:
        local_index = int(torch.argmax(allowed_logits).item())
    else:
        probabilities = torch.softmax(allowed_logits, dim=0)
        local_index = int(torch.multinomial(probabilities, num_samples=1).item())
    return int(ALLOWED_ACTIONS[local_index])


@torch.no_grad()
def model_entropy(model, obs):
    model.eval()
    logits = model(obs_to_tensor(obs, next(model.parameters()).device))[0]
    allowed = torch.tensor(ALLOWED_ACTIONS, dtype=torch.long, device=logits.device)
    allowed_logits = logits.index_select(0, allowed)
    log_probs = torch.log_softmax(allowed_logits, dim=0)
    probs = log_probs.exp()
    entropy = -(probs * log_probs).sum()
    return float(entropy.item())

In [ ]:
train_loader, val_loader = make_data_loaders(
    expert_data["obs"],
    expert_data["actions"],
    seed=SEED,
)

bc_model = SmallMinigridPolicy().to(DEVICE)
bc_history = train_policy(
    bc_model,
    train_loader,
    val_loader,
    epochs=BC_EPOCHS,
    lr=BC_LR,
    desc="BC",
)

plot_training_curves(bc_history, title="Behavior cloning training")

bc_eval = evaluate_policy(
    lambda obs, env: model_action(bc_model, obs, deterministic=True),
    n_episodes=EVAL_EPISODES,
    seed_start=3000,
    record_positions=True,
)

print(f"Best validation accuracy: {bc_history['best_val_acc'] * 100:.1f}%")
print(f"BC success rate: {bc_eval['success_rate'] * 100:.1f}%")
print(f"BC average return: {bc_eval['avg_return']:.3f}")
print(f"BC average episode length: {bc_eval['avg_length']:.1f}")
if bc_eval["success_rate"] < 0.70:
    print(
        "Warning: BC success is much lower than the reference-scale run. "
        "Confirm that AGENT_VIEW_SIZE=15, the full 1,500-episode dataset was generated, "
        "and all cells were run from a fresh kernel."
    )

## 4. Rolling out BC and showing where it fails

The learner now acts in the real environment. We search over seeds for a BC failure case and display the first failure found.

The main failure mode is distribution mismatch: small supervised errors can move the learner into states that were rare or absent in the expert demonstrations. Once there, the policy has less useful training signal.


In [ ]:
def find_seed_for_outcome(policy_fn, target_success, seed_start=4000, n_search=100):
    target_success = bool(target_success)
    for offset in tqdm(range(int(n_search)), desc="searching rollout seeds", leave=False):
        seed = int(seed_start + offset)
        rollout = rollout_policy(policy_fn, seed=seed)
        if rollout["success"] == target_success:
            return seed
    return None


bc_policy_fn = lambda obs, env: model_action(bc_model, obs, deterministic=True)
bc_failure_seed = find_seed_for_outcome(
    bc_policy_fn,
    target_success=False,
    seed_start=4000,
    n_search=300,
)

assert bc_failure_seed is not None, "Expected to find a BC failure seed. Increase n_search or use a larger DoorKey variant."
print(f"BC failure found automatically at seed {bc_failure_seed}.")

bc_rollout = rollout_policy(
    bc_policy_fn,
    seed=bc_failure_seed,
    gif_path=OUTPUT_DIR / "bc_failure.gif",
)
print({
    "return": round(bc_rollout["return"], 3),
    "success": bc_rollout["success"],
    "length": bc_rollout["length"],
})
display_gif(bc_rollout["gif_path"])

fig, ax = plt.subplots(figsize=(4.8, 4.8))
plot_visit_heatmap(
    bc_eval["positions"],
    ax=ax,
    title="BC visited-state heatmap",
    cmap="OrRd",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "bc_visitation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 5. Implementing DAgger

DAgger fixes the dataset mismatch by asking the expert to label states visited by the learner:

$$
D \leftarrow D \cup \{(s, \pi_E(s)) : s \text{ visited by } \pi_\theta\}.
$$

Then we continue training the policy on the aggregated dataset. We track both success rate and the number of expert labels used.


In [ ]:
def collect_dagger_labels(model, n_rollouts, seed_start, max_steps=MAX_STEPS_PER_EPISODE,
                          stochastic_rollout=False):
    images = []
    expert_actions = []
    positions = []
    symbolic_states = []
    rollout_ids = []
    timesteps = []

    for rollout_id in tqdm(range(int(n_rollouts)), desc="full DAgger collection", leave=False):
        env = make_env(render_mode="rgb_array")
        try:
            obs, _ = reset_env(env, seed=int(seed_start + rollout_id))
            layout = build_layout(env)

            for timestep in range(int(max_steps)):
                state = symbolic_state_from_env(env, layout)
                expert_action = oracle_action_from_symbolic_state(layout, state)
                if expert_action == DONE:
                    break

                # Full DAgger keeps every occurrence, including exact repeats.
                images.append(obs_image(obs).copy())
                expert_actions.append(int(expert_action))
                positions.append(get_agent_pos(env))
                symbolic_states.append(state)
                rollout_ids.append(rollout_id)
                timesteps.append(timestep)

                learner_action = model_action(
                    model,
                    obs,
                    deterministic=not stochastic_rollout,
                )
                obs, reward, done, info = step_env(env, learner_action)
                if done:
                    break
        finally:
            env.close()

    return {
        "obs": np.asarray(images, dtype=np.uint8),
        "actions": np.asarray(expert_actions, dtype=np.int64),
        "positions": np.asarray(positions, dtype=np.int16),
        "symbolic_states": np.asarray(symbolic_states, dtype=object),
        "rollout_ids": np.asarray(rollout_ids, dtype=np.int32),
        "timesteps": np.asarray(timesteps, dtype=np.int32),
    }


def evaluation_rank(metrics):
    return (
        float(metrics["success_rate"]),
        float(metrics["avg_return"]),
        -float(metrics["avg_length"]),
    )


initial_label_count = int(len(expert_data["actions"]))
dag_model = copy.deepcopy(bc_model).to(DEVICE)
dag_images = expert_data["obs"].copy()
dag_actions = expert_data["actions"].copy()
dag_additional_labels = 0

dagger_history = [{
    "iteration": 0,
    "labels": initial_label_count,
    "initial_labels": initial_label_count,
    "additional_labels": 0,
    "success_rate": bc_eval["success_rate"],
    "avg_return": bc_eval["avg_return"],
    "avg_length": bc_eval["avg_length"],
}]
dagger_best_model = copy.deepcopy(dag_model).to(DEVICE)
dagger_best_eval = copy.deepcopy(bc_eval)
dagger_best_iteration = 0
dagger_best_labels = initial_label_count

for iteration in range(1, DAGGER_ITERS + 1):
    new_data = collect_dagger_labels(
        dag_model,
        n_rollouts=DAGGER_ROLLOUTS_PER_ITER,
        seed_start=5000 + 1000 * iteration,
    )
    if len(new_data["actions"]) == 0:
        raise RuntimeError("Full DAgger did not collect any learner-visited states.")

    dag_images = np.concatenate([dag_images, new_data["obs"]], axis=0)
    dag_actions = np.concatenate([dag_actions, new_data["actions"]], axis=0)
    dag_additional_labels += int(len(new_data["actions"]))

    train_loader, val_loader = make_data_loaders(
        dag_images,
        dag_actions,
        seed=SEED + 100 + iteration,
    )
    _ = train_policy(
        dag_model,
        train_loader,
        val_loader,
        epochs=DAGGER_EPOCHS_PER_ITER,
        lr=DAGGER_LR,
        desc=f"full DAgger {iteration}/{DAGGER_ITERS}",
    )

    current_eval = evaluate_policy(
        lambda obs, env, model=dag_model: model_action(model, obs, deterministic=True),
        n_episodes=EVAL_EPISODES,
        seed_start=3000,
        record_positions=True,
    )
    row = {
        "iteration": iteration,
        "labels": initial_label_count + dag_additional_labels,
        "initial_labels": initial_label_count,
        "additional_labels": dag_additional_labels,
        "success_rate": current_eval["success_rate"],
        "avg_return": current_eval["avg_return"],
        "avg_length": current_eval["avg_length"],
    }
    dagger_history.append(row)

    if evaluation_rank(current_eval) > evaluation_rank(dagger_best_eval):
        dagger_best_model = copy.deepcopy(dag_model).to(DEVICE)
        dagger_best_eval = copy.deepcopy(current_eval)
        dagger_best_iteration = iteration
        dagger_best_labels = row["labels"]

    print(
        f"Full DAgger iteration {iteration}: "
        f"current success={100 * current_eval['success_rate']:.1f}%, "
        f"cumulative labels={row['labels']} "
        f"({initial_label_count} initial + {dag_additional_labels} additional)"
    )

print(f"Best full-DAgger checkpoint: iteration {dagger_best_iteration}")
dag_success_seed = find_seed_for_outcome(
    lambda obs, env: model_action(dagger_best_model, obs, deterministic=True),
    target_success=True,
    seed_start=3000,
    n_search=EVAL_EPISODES,
)
if dag_success_seed is None:
    dag_success_seed = 3000

dagger_rollout = rollout_policy(
    lambda obs, env: model_action(dagger_best_model, obs, deterministic=True),
    seed=dag_success_seed,
    gif_path=OUTPUT_DIR / "full_dagger_best.gif",
)
display_gif(dagger_rollout["gif_path"])

fig, ax = plt.subplots(figsize=(4.8, 4.8))
plot_visit_heatmap(
    dagger_best_eval["positions"],
    ax=ax,
    title=f"Full DAgger best heatmap (iteration {dagger_best_iteration})",
    cmap="Blues",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "full_dagger_visitation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Budgeted critical-state DAgger

Full DAgger can spend many labels. Here we roll out the learner, score visited states by policy entropy, and query the expert only on the top-k most uncertain states:

$$
H(\pi(\cdot\mid s)) = -\sum_a \pi(a\mid s)\log \pi(a\mid s).
$$

This is a simple active-learning version of DAgger: label the states where the policy is least sure.


In [ ]:
def collect_uncertain_candidates(model, n_rollouts, seed_start, max_steps=MAX_STEPS_PER_EPISODE,
                                 stochastic_rollout=False):
    candidates = []

    for rollout_id in tqdm(range(int(n_rollouts)), desc="critical-state collection", leave=False):
        env = make_env(render_mode="rgb_array")
        seed = int(seed_start + rollout_id)
        try:
            obs, _ = reset_env(env, seed=seed)
            layout = build_layout(env)
            layout_signature = (
                layout.width,
                layout.height,
                tuple(sorted(layout.walls)),
                layout.key_pos,
                layout.door_pos,
                layout.goal_pos,
            )

            for timestep in range(int(max_steps)):
                state = symbolic_state_from_env(env, layout)
                image = obs_image(obs).copy()
                entropy = model_entropy(model, obs)

                # Deduplication combines exact image, full symbolic state, and fixed map.
                dedup_key = (layout_signature, state, image.tobytes())
                candidates.append({
                    "obs": image,
                    "state": state,
                    "layout": layout,
                    "position": get_agent_pos(env),
                    "entropy": float(entropy),
                    "seed": seed,
                    "rollout_id": int(rollout_id),
                    "timestep": int(timestep),
                    "dedup_key": dedup_key,
                })

                learner_action = model_action(
                    model,
                    obs,
                    deterministic=not stochastic_rollout,
                )
                obs, reward, done, info = step_env(env, learner_action)
                if done:
                    break
        finally:
            env.close()

    return candidates


def label_top_uncertain_states(candidates, k):
    unique = {}
    for candidate in candidates:
        if candidate["dedup_key"] not in unique:
            unique[candidate["dedup_key"]] = candidate

    ranked = sorted(
        unique.values(),
        key=lambda candidate: (
            -candidate["entropy"],
            candidate["seed"],
            candidate["rollout_id"],
            candidate["timestep"],
        ),
    )
    selected = ranked[:min(int(k), len(ranked))]

    images = []
    actions = []
    positions = []
    entropies = []
    for candidate in selected:
        expert_action = oracle_action_from_symbolic_state(
            candidate["layout"],
            candidate["state"],
        )
        if expert_action == DONE:
            continue
        images.append(candidate["obs"])
        actions.append(int(expert_action))
        positions.append(candidate["position"])
        entropies.append(candidate["entropy"])

    return {
        "obs": np.asarray(images, dtype=np.uint8),
        "actions": np.asarray(actions, dtype=np.int64),
        "positions": np.asarray(positions, dtype=np.int16),
        "entropies": np.asarray(entropies, dtype=np.float32),
        "candidate_count": int(len(candidates)),
        "unique_count": int(len(unique)),
        "selected_count": int(len(actions)),
    }


print(
    "Critical-state deduplication: exact image + complete symbolic state + fixed map; "
    "within the current iteration only. Entropy ties use seed/rollout/timestep order."
)
critical_model = copy.deepcopy(bc_model).to(DEVICE)
critical_images = expert_data["obs"].copy()
critical_actions = expert_data["actions"].copy()
critical_additional_labels = 0

critical_history = [{
    "iteration": 0,
    "labels": initial_label_count,
    "initial_labels": initial_label_count,
    "additional_labels": 0,
    "success_rate": bc_eval["success_rate"],
    "avg_return": bc_eval["avg_return"],
    "avg_length": bc_eval["avg_length"],
}]
critical_best_model = copy.deepcopy(critical_model).to(DEVICE)
critical_best_eval = copy.deepcopy(bc_eval)
critical_best_iteration = 0
critical_best_labels = initial_label_count

for iteration in range(1, CRITICAL_ITERS + 1):
    candidates = collect_uncertain_candidates(
        critical_model,
        n_rollouts=CRITICAL_ROLLOUTS_PER_ITER,
        seed_start=9000 + 1000 * iteration,
    )
    selected_data = label_top_uncertain_states(
        candidates,
        k=CRITICAL_LABELS_PER_ITER,
    )
    if len(selected_data["actions"]) == 0:
        raise RuntimeError("Critical-state DAgger did not select any queryable states.")
    if selected_data["selected_count"] < CRITICAL_LABELS_PER_ITER:
        print(
            f"Critical DAgger iteration {iteration}: only "
            f"{selected_data['selected_count']} unique queryable states were available "
            f"for a budget of {CRITICAL_LABELS_PER_ITER}."
        )

    critical_images = np.concatenate([critical_images, selected_data["obs"]], axis=0)
    critical_actions = np.concatenate([critical_actions, selected_data["actions"]], axis=0)
    critical_additional_labels += int(len(selected_data["actions"]))

    train_loader, val_loader = make_data_loaders(
        critical_images,
        critical_actions,
        seed=SEED + 200 + iteration,
    )
    _ = train_policy(
        critical_model,
        train_loader,
        val_loader,
        epochs=CRITICAL_EPOCHS_PER_ITER,
        lr=CRITICAL_LR,
        desc=f"critical DAgger {iteration}/{CRITICAL_ITERS}",
    )

    current_eval = evaluate_policy(
        lambda obs, env, model=critical_model: model_action(model, obs, deterministic=True),
        n_episodes=EVAL_EPISODES,
        seed_start=3000,
        record_positions=True,
    )
    row = {
        "iteration": iteration,
        "labels": initial_label_count + critical_additional_labels,
        "initial_labels": initial_label_count,
        "additional_labels": critical_additional_labels,
        "success_rate": current_eval["success_rate"],
        "avg_return": current_eval["avg_return"],
        "avg_length": current_eval["avg_length"],
    }
    critical_history.append(row)

    if evaluation_rank(current_eval) > evaluation_rank(critical_best_eval):
        critical_best_model = copy.deepcopy(critical_model).to(DEVICE)
        critical_best_eval = copy.deepcopy(current_eval)
        critical_best_iteration = iteration
        critical_best_labels = row["labels"]

    print(
        f"Critical DAgger iteration {iteration}: "
        f"candidates={selected_data['candidate_count']}, "
        f"unique={selected_data['unique_count']}, "
        f"queried={selected_data['selected_count']}, "
        f"current success={100 * current_eval['success_rate']:.1f}%, "
        f"cumulative labels={row['labels']} "
        f"({initial_label_count} initial + {critical_additional_labels} additional)"
    )

print(f"Best critical-DAgger checkpoint: iteration {critical_best_iteration}")
critical_success_seed = find_seed_for_outcome(
    lambda obs, env: model_action(critical_best_model, obs, deterministic=True),
    target_success=True,
    seed_start=3000,
    n_search=EVAL_EPISODES,
)
if critical_success_seed is None:
    critical_success_seed = 3000

critical_rollout = rollout_policy(
    lambda obs, env: model_action(critical_best_model, obs, deterministic=True),
    seed=critical_success_seed,
    gif_path=OUTPUT_DIR / "critical_dagger_best.gif",
)
display_gif(critical_rollout["gif_path"])

fig, ax = plt.subplots(figsize=(4.8, 4.8))
plot_visit_heatmap(
    critical_best_eval["positions"],
    ax=ax,
    title=f"Critical DAgger best heatmap (iteration {critical_best_iteration})",
    cmap="Purples",
)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "critical_dagger_visitation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Final comparison and visual summary

We finish with label efficiency, final heatmaps, and a compact metrics table. The curves use expert-label count on the x-axis, so BC starts with the demonstration dataset and DAgger methods move right as they request more labels.


In [ ]:
# This is a helper function to plot success rate versus expert-label count. You may use it later. No need to change it.
def plot_label_efficiency(curves):
    fig = plt.figure(figsize=(6.5, 4))
    for name, history, marker in curves:
        labels = [row["labels"] for row in history]
        success = [100.0 * row["success_rate"] for row in history]
        plt.plot(labels, success, marker=marker, linewidth=2, label=name)
    plt.xlabel("cumulative expert labels used")
    plt.ylabel("success rate (%)")
    plt.title("Current-iterate success versus label count")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "success_vs_expert_labels.png", dpi=160, bbox_inches="tight")
    plt.show()


# This is a helper function to display a compact Markdown results table. You may use it later. No need to change it.
def display_results_table(rows):
    lines = [
        "| method | expert labels used | success rate | average return | average episode length |",
        "|---|---:|---:|---:|---:|",
    ]
    for row in rows:
        lines.append(
            f"| {row['method']} | {row['labels']} | "
            f"{row['success_rate'] * 100:.1f}% | "
            f"{row['avg_return']:.3f} | "
            f"{row['avg_length']:.1f} |"
        )
    display(Markdown("\n".join(lines)))

bc_curve = [{
    "labels": initial_label_count,
    "success_rate": bc_eval["success_rate"],
}]
plot_label_efficiency([
    ("BC", bc_curve, "o"),
    ("Full DAgger", dagger_history, "s"),
    ("Critical-state DAgger", critical_history, "^"),
])

print("The success-versus-label curves show current iterates.")
print(
    "Best-checkpoint criterion for both DAgger variants: "
    "higher success rate, then higher return, then shorter episode length."
)
print(
    f"Full DAgger labels: {initial_label_count} initial + "
    f"{dagger_history[-1]['additional_labels']} additional = {dagger_history[-1]['labels']} total"
)
print(
    f"Critical DAgger labels: {initial_label_count} initial + "
    f"{critical_history[-1]['additional_labels']} additional = {critical_history[-1]['labels']} total"
)

comparison_rows = [
    {
        "method": "Behavior cloning",
        "labels": initial_label_count,
        "success_rate": bc_eval["success_rate"],
        "avg_return": bc_eval["avg_return"],
        "avg_length": bc_eval["avg_length"],
    },
    {
        "method": f"Full DAgger (best iter. {dagger_best_iteration})",
        "labels": dagger_best_labels,
        "success_rate": dagger_best_eval["success_rate"],
        "avg_return": dagger_best_eval["avg_return"],
        "avg_length": dagger_best_eval["avg_length"],
    },
    {
        "method": f"Critical DAgger (best iter. {critical_best_iteration})",
        "labels": critical_best_labels,
        "success_rate": critical_best_eval["success_rate"],
        "avg_return": critical_best_eval["avg_return"],
        "avg_length": critical_best_eval["avg_length"],
    },
]
display_results_table(comparison_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
plot_visit_heatmap(
    bc_eval["positions"],
    ax=axes[0],
    title="BC",
    cmap="OrRd",
    show_colorbar=False,
)
plot_visit_heatmap(
    dagger_best_eval["positions"],
    ax=axes[1],
    title=f"Full DAgger best (iter. {dagger_best_iteration})",
    cmap="Blues",
    show_colorbar=False,
)
plot_visit_heatmap(
    critical_best_eval["positions"],
    ax=axes[2],
    title=f"Critical DAgger best (iter. {critical_best_iteration})",
    cmap="Purples",
    show_colorbar=False,
)
fig.suptitle("Matched-seed state-visitation heatmaps")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "final_visitation_heatmaps.png", dpi=160, bbox_inches="tight")
plt.show()

print("Representative GIFs:")
print(" -", bc_rollout["gif_path"])
print(" -", dagger_rollout["gif_path"])
print(" -", critical_rollout["gif_path"])
print("All plots and GIFs are available under", OUTPUT_DIR.resolve())

## Reflection

The experiment shows the basic imitation-learning pattern. BC is simple and often learns useful behavior from expert trajectories, but it can suffer from distribution shift when its own early mistakes create unfamiliar states. DAgger reduces that mismatch by adding expert labels for learner-visited states. Critical-state DAgger keeps the same idea but tries to spend expert labels where the current policy is most uncertain, trading exhaustive labeling for a smaller, more focused query budget.


# 8. Answers to Questions 4 and 5

## Question 4: Behavior Cloning, Distribution Shift, and DAgger

### 1.

The BC model reached 97.8 percent validation accuracy, but its rollout success rate was 96 percent. High validation accuracy only shows that the model works well on states taken from expert paths.

During a real rollout, one wrong action can move the agent away from the expert path. The model may then see states that were not common in its training data. More mistakes can follow, and the agent may fail. For example, the detected BC failure ran for the full 768 steps without reaching the goal.

### 2.

DAgger collects states visited by the learner and asks the expert for the correct action in those states. This adds examples of mistakes and recovery actions to the training data.

Collecting more expert-only paths mostly adds more examples of correct behavior. It does not teach the learner what to do after it leaves the correct path. Learner-visited states are therefore more useful for learning recovery behavior.

### 3.

The BC failure GIF shows the agent staying in a small area for most of the episode. The failure heatmap also has a very dark area near the same cells. This is evidence of repeated local behavior after the agent moved away from a useful path.

The matched-seed heatmaps show that successful BC and DAgger policies mostly use the same main route. There is limited evidence of a completely new route. Full DAgger mainly improved reliability and speed.

BC had a 96 percent success rate and an average episode length of 65.9 steps. The best Full DAgger model reached 97 percent success and reduced the average length to 59.9 steps. Its average return also increased from 0.948 to 0.957.

### 4.

The oracle can use the full environment state to find the correct next action. That action can then be used as a label for the image seen by the learner.

A problem can happen when the same learner image appears in two different hidden states. For example, the unseen map, key possession, or door state may be different. The oracle may choose different actions even though the learner receives the same image. This can create conflicting training examples.

## Question 5: Critical-State DAgger and Expert-Label Efficiency

### 1.

High entropy means that the policy is unsure which action to choose. Asking the expert about these states can be useful because the policy may need more training there.

However, entropy can fail when the policy is confidently wrong. In that case, the policy gives one wrong action a high score and has low entropy. The state is then not selected even though it is important.

### 2.

BC used 50,204 expert labels and reached 96 percent success.

The best Full DAgger checkpoint used 53,090 labels. This was 2,886 extra labels. It reached 97 percent success and reduced the average episode length from 65.9 to 59.9 steps.

Critical DAgger used 59,804 labels after all rounds, which was 9,600 extra labels. Its best checkpoint was still the starting BC model, with 96 percent success and an average length of 65.9 steps.

Therefore, Critical DAgger did not match the best Full DAgger result with fewer labels. Its selected high-entropy states did not provide enough useful new coverage in this run.

The current Full DAgger success rate changed between 86 and 97 percent and ended at 94 percent. Critical DAgger stayed between 95 and 96 percent and ended at 96 percent. This shows that later training rounds do not always produce a better model, so keeping the best checkpoint is important.

### 3.

The notebook removes duplicates using the exact image, full symbolic state, and fixed map. Duplicate removal is done only inside the current round.

This avoids spending several labels on exactly the same state during one round. However, the same state may still be queried again in a later round.

Using exact images is simple, but similar states may still be treated as different. Using symbolic states gives a clearer state description, but it uses information that the learner does not normally see. Grouping larger grid areas saves more labels, but it may combine states that need different actions.

### 4.

One alternative is ensemble disagreement. Several policies can be trained with different random starting settings. The expert is queried when these policies choose different actions for the same state.

This method needs more training time and memory because several models must be saved and run. It can help find states where one policy is confidently wrong but the other policies disagree, which entropy from a single policy may miss.